# ProSIT 2 — Feature Engineering V2 (train.csv only)

**Goal:** auditable, leakage-conscious and reproducible feature engineering for loan-level classification and regression, with a separate *descriptive-only* lifetime dataset.

**Design decisions:** original ID preservation; fail-fast source-ID checks; quarantine ambiguous loan IDs (never invent customer IDs); strict *earlier calendar day* borrowing history; repayment outcomes only in an optional sensitivity feature set; no global categorical encoding, imputation or resampling. All outputs go to `data/processed_v2/` and `audits/v2/`.

**Important limitations:** `disbursement_date` is a proxy for the application decision date. `due_date` is **not** a confirmed outcome-observation timestamp. The training data does **not** establish suitable/affordable loan limits; the regression target is historical disbursed amount. A lender or country ID may be useful for audit, but it is **not** in the primary predictor list. This notebook never reads `test.csv`, FRED, or Findex.

**Run instructions:** put this notebook at the ProSIT2 project root with `data/raw/train.csv` (or set `TRAIN_PATH` explicitly), then Run All in VS Code/Jupyter. Review quarantine and audit outputs before fitting any model.


## 0 — Configuration, reproducibility and dependencies

In [ ]:
from pathlib import Path
from collections import Counter
import json, hashlib, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
TRAIN_PATH = Path('data/raw/train.csv')
if not TRAIN_PATH.exists():
    TRAIN_PATH = Path('train.csv')  # convenient local fallback; still ONLY train.csv
OUT = Path('data/processed_v2')
AUDIT = Path('audits/v2')
FIGURES = OUT / 'figures'
for directory in [OUT, AUDIT, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

# Turn off strict format checking ONLY after independently verifying the source schema.
ENFORCE_SIX_DIGIT_CUSTOMER_IDS = True
# Strict policy: any non-unique nonmissing tbl_loan_id is excluded from *modelling*
# and from history engineering (with full original records preserved in an audit).
QUARANTINE_REPEATED_LOAN_IDS = True
# No target-derived repayment history in PRIMARY feature set, irrespective of this switch.
BUILD_SENSITIVITY_REPAYMENT_FEATURES = True

if not TRAIN_PATH.is_file():
    raise FileNotFoundError(f'Missing train.csv: {TRAIN_PATH.resolve()}')
print('Input:', TRAIN_PATH.resolve())
print('Versions:', {'python':sys.version.split()[0], 'pandas':pd.__version__, 'numpy':np.__version__})


## 1 — Read the original identifiers as strings and audit the source

In [ ]:
REQUIRED = [
    'ID', 'customer_id', 'country_id', 'tbl_loan_id', 'Total_Amount',
    'Total_Amount_to_Repay', 'loan_type', 'disbursement_date', 'duration',
    'lender_id', 'New_versus_Repeat', 'Amount_Funded_By_Lender',
    'Lender_portion_Funded', 'due_date', 'Lender_portion_to_be_repaid', 'target'
]
ID_COLS = ['ID','customer_id','tbl_loan_id']
CAT_COLS = ['country_id','loan_type','lender_id','New_versus_Repeat']
read_as_text = {col: 'string' for col in ID_COLS + CAT_COLS}
source = pd.read_csv(TRAIN_PATH, dtype=read_as_text, low_memory=False)
missing = sorted(set(REQUIRED) - set(source.columns))
if missing:
    raise ValueError(f'Source missing required variables: {missing}')
source = source[REQUIRED].copy()
source['_source_row'] = np.arange(len(source), dtype='int64')
for col in ID_COLS + CAT_COLS:
    source[col] = source[col].str.strip().replace('', pd.NA)

customer_valid = source.customer_id.notna()
if ENFORCE_SIX_DIGIT_CUSTOMER_IDS:
    customer_valid &= source.customer_id.str.fullmatch(r'[1-9][0-9]{5}').fillna(False)
else:
    customer_valid &= source.customer_id.str.fullmatch(r'\d+').fillna(False)
source_id_report = pd.DataFrame({
    'field': ID_COLS,
    'missing': [int(source[c].isna().sum()) for c in ID_COLS],
    'distinct': [int(source[c].nunique()) for c in ID_COLS]
})
source_id_report.to_csv(AUDIT / 'source_identifier_summary.csv', index=False)
source.loc[~customer_valid, ['_source_row','ID','customer_id','tbl_loan_id']].to_csv(AUDIT/'invalid_customer_ids.csv', index=False)
if not customer_valid.all():
    examples = source.loc[~customer_valid, ['_source_row','ID','customer_id']].head(12)
    raise AssertionError('Source customer IDs violate configured format. See audits/v2/invalid_customer_ids.csv.\n' + examples.to_string(index=False))
if source['ID'].isna().any():
    raise AssertionError('Missing source ID; cannot guarantee record traceability.')
print('Source rows:', len(source),'distinct customers:',source.customer_id.nunique())
print('Sample preserved customer IDs:',source.customer_id.head().tolist())
display(source_id_report)


## 2 — Parse types and audit repeated identifiers

**We do not silently keep conflicting records.** Exact duplicate full rows are collapsed with an audit; repeated `ID` after that is a hard error; repeated nonmissing `tbl_loan_id` is quarantined by default because it may inflate borrower history. Original `train.csv` is unchanged.


In [ ]:
work = source.copy()
for c in ['disbursement_date','due_date']:
    work[c] = pd.to_datetime(work[c], errors='coerce')
for c in ['Total_Amount','Total_Amount_to_Repay','duration','Amount_Funded_By_Lender','Lender_portion_Funded','Lender_portion_to_be_repaid','target']:
    work[c] = pd.to_numeric(work[c], errors='coerce')

exact_cols = [c for c in REQUIRED]  # after parsing, full content equality
exact_mask = work.duplicated(subset=exact_cols, keep='first')
work.loc[exact_mask].to_csv(AUDIT/'exact_duplicate_rows_removed.csv', index=False)
work = work.loc[~exact_mask].copy()
if work.ID.duplicated(keep=False).any():
    dupe = work.loc[work.ID.duplicated(keep=False)]
    dupe.to_csv(AUDIT/'conflicting_source_ID.csv', index=False)
    raise AssertionError('Source ID is not unique after exact duplicates; see audits/v2/conflicting_source_ID.csv')

repeat = work.tbl_loan_id.notna() & work.tbl_loan_id.duplicated(keep=False)
work.loc[repeat].to_csv(AUDIT/'repeated_loan_ids_review.csv', index=False)
if QUARANTINE_REPEATED_LOAN_IDS:
    clean = work.loc[~repeat].copy()
    quarantine = work.loc[repeat].copy()
else:
    if repeat.any():
        raise AssertionError('Repeated loan IDs exist: enable quarantine or resolve conflicts manually before continuing.')
    clean = work.copy(); quarantine = work.iloc[0:0].copy()

# Fail closed on invalid binary labels; do not impute targets.
invalid_target = ~clean.target.isin([0, 1])
clean.loc[invalid_target].to_csv(AUDIT/'invalid_binary_targets.csv', index=False)
if invalid_target.any():
    raise AssertionError('Invalid targets. Review audits/v2/invalid_binary_targets.csv')
clean['target'] = clean.target.astype('int8')
# For predictive features, invalid historical amounts and durations are missing, not zero.
clean.loc[clean.Total_Amount.le(0) | ~np.isfinite(clean.Total_Amount), 'Total_Amount'] = np.nan
clean.loc[clean.duration.le(0) | ~np.isfinite(clean.duration), 'duration'] = np.nan
# An outcome from earlier loan whose due_date < disbursement cannot be ruled out,
# but chronology is suspicious; exclude such outcome dates from sensitivity features.
suspect_due = clean.due_date.notna() & clean.disbursement_date.notna() & (clean.due_date < clean.disbursement_date)
clean.loc[suspect_due].to_csv(AUDIT/'invalid_due_before_disbursement.csv', index=False)
clean.loc[suspect_due,'due_date'] = pd.NaT

quality = {
  'source_rows':int(len(source)), 'exact_duplicates_removed':int(exact_mask.sum()),
  'repeated_loan_id_records_quarantined':int(len(quarantine)),
  'retained_loan_rows':int(len(clean)),
  'retained_unique_customers':int(clean.customer_id.nunique()),
  'missing_disbursement_dates':int(clean.disbursement_date.isna().sum()),
  'missing_due_dates_after_audit':int(clean.due_date.isna().sum()),
  'invalid_due_dates_nullified':int(suspect_due.sum()),
  'retained_default_count':int(clean.target.sum()),
  'retained_default_rate':float(clean.target.mean()) if len(clean) else None,
}
(AUDIT/'source_quality_summary.json').write_text(json.dumps(quality,indent=2))
print(json.dumps(quality,indent=2))
assert len(clean)>0, 'No rows retained after integrity checks.'


## 3 — Identity invariants: prove every surviving record belongs to the original customer

In [ ]:
identity = clean[['ID','customer_id','tbl_loan_id']].merge(
    source[['ID','customer_id','tbl_loan_id']].rename(columns={
        'customer_id':'customer_id_original','tbl_loan_id':'tbl_loan_id_original'
    }), on='ID', how='left',validate='one_to_one',indicator=True
)
bad_identity = identity.loc[
    identity._merge.ne('both') |
    identity.customer_id.ne(identity.customer_id_original).fillna(True) |
    (~identity.tbl_loan_id.eq(identity.tbl_loan_id_original).fillna(False) &
     ~(identity.tbl_loan_id.isna() & identity.tbl_loan_id_original.isna()))
]
bad_identity.to_csv(AUDIT/'identity_mismatches.csv', index=False)
assert bad_identity.empty, 'Source-to-clean identifier mismatch: stop, inspect identity_mismatches.csv'
assert clean.customer_id.str.fullmatch(r'[1-9][0-9]{5}').all() if ENFORCE_SIX_DIGIT_CUSTOMER_IDS else clean.customer_id.notna().all()
print('PASS: ID -> customer_id and tbl_loan_id mapping preserved for',len(clean),'retained source records.')


## 4 — Feature policy

We create three **explicitly separate** candidate feature groups:

- `F0_context`: known current application terms (`loan_type`, `duration`) and calendar proxies.
- `F1_borrowing_primary`: F0 + prior **disbursed loan events** only. This is the **primary** model family.
- `F2_outcome_sensitivity`: F1 + prior outcomes eligible under the **unverified due-date proxy**; exploratory only.

`country_id` and `lender_id` are retained for audits, **not primary predictors**. Current amounts and current labels are targets only. Global one-hot encoding is intentionally absent: fit encoders/imputers within training folds in the modelling notebooks.


## 5 — Dataset A: lifetime descriptive customer profile (never feed into loan-level models)

In [ ]:
life = clean.copy()
life['_is_default'] = life.target.eq(1).astype('int8')
life['_is_repaid'] = life.target.eq(0).astype('int8')
lifetime = life.groupby('customer_id',dropna=False).agg(
    lifetime_loan_count=('ID','size'),
    lifetime_unique_loan_ids=('tbl_loan_id','nunique'),
    lifetime_total_disbursed=('Total_Amount', lambda s:s.sum(min_count=1)),
    lifetime_mean_disbursed=('Total_Amount','mean'),
    lifetime_median_disbursed=('Total_Amount','median'),
    lifetime_mean_duration=('duration','mean'),
    first_disbursement_date=('disbursement_date','min'),
    last_disbursement_date=('disbursement_date','max'),
    lifetime_defaults=('_is_default','sum'),
    lifetime_repaid=('_is_repaid','sum'),
    distinct_loan_types=('loan_type','nunique'),
    distinct_lenders=('lender_id','nunique'),
).reset_index()
lifetime['lifetime_default_rate'] = lifetime.lifetime_defaults/lifetime.lifetime_loan_count.replace(0,np.nan)
lifetime['lifetime_tenure_days'] = (lifetime.last_disbursement_date-lifetime.first_disbursement_date).dt.days
assert lifetime.customer_id.is_unique
assert set(lifetime.customer_id)==set(clean.customer_id)
assert lifetime.lifetime_loan_count.sum()==len(clean)
print('Dataset A:',lifetime.shape)
display(lifetime.head())


## 6 — Dataset B: safe, point-in-time loan snapshots

Loan records are one row each. For each assessment date `t`, borrow history is constructed from loans of the same customer whose **calendar date** is strictly before `t`. Loans disbursed on the same date do not contribute to one another. A loan event can be counted even if its outcome is unknown.

For optional repayment features, the prior loan's due date must be **strictly earlier** than `t`, but this does not prove the outcome label was actually known by that date. Avoid using F2 for primary claims. Ambiguous same-day prior-event ordering is treated as undefined for `previous_most_recent_loan_amount`.


In [ ]:
BORROW_FEATURES = [
    'previous_loan_count','previous_valid_amount_count','previous_total_amount_borrowed',
    'previous_mean_loan_amount','previous_min_loan_amount','previous_max_loan_amount',
    'previous_mean_duration','days_since_previous_loan','previous_loans_last_90d',
    'previous_loans_last_180d','previous_mean_days_between_loans',
    'previous_mean_amount_last_3','previous_most_recent_loan_amount',
    'previous_amount_trend_last_3'
]
BORROW_FLAGS = ['has_previous_loan','has_previous_valid_amount','has_previous_valid_duration','has_multiple_previous_loan_dates']
REPAY_FEATURES = [
    'previous_known_outcome_count','previous_default_count','previous_repaid_count',
    'previous_default_rate','previous_repayment_rate','days_since_last_default',
    'most_recent_known_outcome','defaults_last_3_known_outcomes',
    'defaults_last_5_known_outcomes','repayment_rate_last_3_known_outcomes',
    'previous_loans_without_known_outcome','has_known_previous_outcome'
]
CALENDAR_FEATURES = ['disbursement_year','disbursement_month','disbursement_quarter',
                     'disbursement_dayofweek','disbursement_day','disbursement_is_weekend']
CONTEXT_FEATURES=['loan_type','duration']
ALL_DERIVED = BORROW_FEATURES + BORROW_FLAGS + REPAY_FEATURES + CALENDAR_FEATURES

# Chronology is at date resolution, not time-of-day. This guarantees same-day exclusion.
loans = clean.sort_values('_source_row').reset_index(drop=True).copy()
loans['row_key'] = np.arange(len(loans),dtype='int64')
loans['_loan_date'] = loans.disbursement_date.dt.normalize()
loans['_due_date'] = loans.due_date.dt.normalize()
for feat in BORROW_FEATURES + REPAY_FEATURES:
    loans[feat] = np.nan
for feat in BORROW_FLAGS + ['has_known_previous_outcome']:
    loans[feat] = 0

# Use source loan records as events. Direct loops are easy to audit for ~69k rows.
def finite_mean(x):
    x=np.asarray(x,dtype=float)
    x=x[np.isfinite(x)]
    return float(x.mean()) if len(x) else np.nan

def engineer_group(g):
    dates=g['_loan_date'].to_numpy(dtype='datetime64[ns]')
    due=g['_due_date'].to_numpy(dtype='datetime64[ns]')
    amounts=g.Total_Amount.to_numpy(dtype=float)
    durations=g.duration.to_numpy(dtype=float)
    labels=g.target.to_numpy(dtype=int)
    indices=g.index.to_numpy()
    dated=~np.isnat(dates)
    result=[]
    for j,idx in enumerate(indices):
        if not dated[j]:
            result.append((idx,{}))
            continue
        t=dates[j]
        p=np.flatnonzero(dated & (dates<t))
        n=len(p)
        features={'previous_loan_count':n,
                  'previous_loans_last_90d':int(((dates[p]>=t-np.timedelta64(90,'D'))).sum()),
                  'previous_loans_last_180d':int(((dates[p]>=t-np.timedelta64(180,'D'))).sum()),
                  'has_previous_loan':int(n>0)}
        if n:
            pa=amounts[p]; pdur=durations[p]
            va=pa[np.isfinite(pa)]; vd=pdur[np.isfinite(pdur)]
            features.update(previous_valid_amount_count=len(va),
                previous_total_amount_borrowed=float(va.sum()) if len(va) else np.nan,
                previous_mean_loan_amount=finite_mean(va),
                previous_min_loan_amount=float(va.min()) if len(va) else np.nan,
                previous_max_loan_amount=float(va.max()) if len(va) else np.nan,
                previous_mean_duration=finite_mean(vd),
                has_previous_valid_amount=int(len(va)>0),
                has_previous_valid_duration=int(len(vd)>0))
            distinct_dates=np.unique(dates[p]); distinct_dates.sort()
            features['days_since_previous_loan']=float((t-distinct_dates[-1])/np.timedelta64(1,'D'))
            features['has_multiple_previous_loan_dates']=int(len(distinct_dates)>1)
            features['previous_mean_days_between_loans']=(
                float(np.diff(distinct_dates).astype('timedelta64[D]').astype(float).mean())
                if len(distinct_dates)>1 else np.nan)
            # Last three events, stable order only across DISTINCT dates.
            ordered=sorted(p, key=lambda k:(dates[k],int(g['_source_row'].iloc[k])))
            last3=np.asarray(ordered[-3:],dtype=int)
            last3valid=amounts[last3][np.isfinite(amounts[last3])]
            features['previous_mean_amount_last_3']=finite_mean(last3valid)
            latest=np.flatnonzero(dates[p]==distinct_dates[-1])
            # If multiple prior loans share the latest date, ordering unknown: leave missing.
            features['previous_most_recent_loan_amount']=(float(amounts[p[latest[0]]])
                if len(latest)==1 and np.isfinite(amounts[p[latest[0]]]) else np.nan)
            # Trend only with unambiguous last3 ordered on distinct dates.
            features['previous_amount_trend_last_3']=(float(amounts[last3[-1]]-amounts[last3[0]])
                if len(last3)>=2 and len(np.unique(dates[last3]))==len(last3)
                and np.isfinite(amounts[last3[0]]) and np.isfinite(amounts[last3[-1]]) else np.nan)
        else:
            features.update(previous_valid_amount_count=0,
                            has_previous_valid_amount=0,has_previous_valid_duration=0,
                            has_multiple_previous_loan_dates=0)
        if BUILD_SENSITIVITY_REPAYMENT_FEATURES:
            # Proxy: outcomes are considered eligible only after their due date.
            eligible=p[(~np.isnat(due[p])) & (due[p]<t)]
            eo=len(eligible)
            f=features
            f['previous_known_outcome_count']=eo
            f['previous_default_count']=int((labels[eligible]==1).sum())
            f['previous_repaid_count']=int((labels[eligible]==0).sum())
            f['has_known_previous_outcome']=int(eo>0)
            f['previous_loans_without_known_outcome']=n-eo
            if eo:
                f['previous_default_rate']=f['previous_default_count']/eo
                f['previous_repayment_rate']=f['previous_repaid_count']/eo
                order=sorted(eligible,key=lambda k:(due[k],dates[k],int(g['_source_row'].iloc[k])))
                recent=np.asarray(order,dtype=int)
                f['most_recent_known_outcome']=int(labels[recent[-1]]) if np.sum(due[recent]==due[recent[-1]])==1 else np.nan
                f['defaults_last_3_known_outcomes']=int((labels[recent[-3:]]==1).sum())
                f['defaults_last_5_known_outcomes']=int((labels[recent[-5:]]==1).sum())
                f['repayment_rate_last_3_known_outcomes']=float((labels[recent[-3:]]==0).mean())
                defaults=eligible[labels[eligible]==1]
                if len(defaults):
                    f['days_since_last_default']=float((t-due[defaults].max())/np.timedelta64(1,'D'))
        result.append((idx,features))
    return result

updates=[]
for _, group in loans.groupby('customer_id',sort=False):
    updates.extend(engineer_group(group))
for idx,features in updates:
    for feature,value in features.items():
        loans.at[idx,feature]=value

# Calendar features from disbursement date as a provisional decision-time proxy.
date=loans['_loan_date'].dt
loans['disbursement_year']=date.year.astype('float64')
loans['disbursement_month']=date.month.astype('float64')
loans['disbursement_quarter']=date.quarter.astype('float64')
loans['disbursement_dayofweek']=date.dayofweek.astype('float64')
loans['disbursement_day']=date.day.astype('float64')
loans['disbursement_is_weekend']=np.where(loans._loan_date.notna(),(date.dayofweek>=5).astype(float),np.nan)
print('Dataset B:',loans.shape)
display(loans[['ID','customer_id','row_key','previous_loan_count','previous_mean_loan_amount','previous_default_rate']].head(10))


## 7 — Strong automated integrity and chronology tests

In [ ]:
# Persist mapping at the source-row level; row_key is NEVER treated as customer_id.
assert loans.row_key.is_unique
assert loans.ID.is_unique
assert set(loans.ID)==set(clean.ID)
mapping=loans[['ID','customer_id']].merge(clean[['ID','customer_id']].rename(columns={'customer_id':'source_customer_id'}),on='ID',validate='one_to_one')
assert mapping.customer_id.eq(mapping.source_customer_id).all()
assert loans.customer_id.str.fullmatch(r'[1-9][0-9]{5}').all() if ENFORCE_SIX_DIGIT_CUSTOMER_IDS else loans.customer_id.notna().all()
assert (loans.loc[loans._loan_date.notna(),'previous_loan_count']>=0).all()
assert (loans.loc[loans._loan_date.notna(),'previous_loans_last_90d']<=loans.loc[loans._loan_date.notna(),'previous_loans_last_180d']).all()
assert (loans.loc[loans._loan_date.notna(),'previous_loans_last_180d']<=loans.loc[loans._loan_date.notna(),'previous_loan_count']).all()
first=loans.groupby('customer_id')._loan_date.transform('min')
earliest=loans._loan_date.notna() & loans._loan_date.eq(first)
assert loans.loc[earliest,'previous_loan_count'].eq(0).all()
assert loans.dropna(subset=['_loan_date']).groupby(['customer_id','_loan_date']).previous_loan_count.nunique().le(1).all()
if BUILD_SENSITIVITY_REPAYMENT_FEATURES:
    eligible_rows=loans._loan_date.notna()
    assert (loans.loc[eligible_rows,'previous_known_outcome_count']<=loans.loc[eligible_rows,'previous_loan_count']).all()
    assert (loans.loc[eligible_rows,'previous_default_count']+loans.loc[eligible_rows,'previous_repaid_count']==loans.loc[eligible_rows,'previous_known_outcome_count']).all()
    for col in ['previous_default_rate','previous_repayment_rate','repayment_rate_last_3_known_outcomes']:
        assert loans[col].dropna().between(0,1).all()
# Histories of undated current loans are undefined (not fabricated zeros).
assert loans.loc[loans._loan_date.isna(),'previous_loan_count'].isna().all()
print('PASS: identity, chronology, same-day exclusion, counts and rates.')


## 8 — Define separate task-specific feature manifests (no global encoding)

**Primary F1** is *context + borrowing history*, without repayment labels, geographic IDs or lender IDs. **F0** includes only application context and calendar. **F2** is an exploratory due-date proxy outcome sensitivity analysis. Full candidate columns remain in the master file so comparison is possible.


In [ ]:
F0_CONTEXT = CONTEXT_FEATURES + CALENDAR_FEATURES
F1_BORROWING_PRIMARY = F0_CONTEXT + BORROW_FEATURES + BORROW_FLAGS
F2_OUTCOME_SENSITIVITY = F1_BORROWING_PRIMARY + REPAY_FEATURES
FEATURE_SETS={
 'F0_context':F0_CONTEXT,
 'F1_borrowing_primary':F1_BORROWING_PRIMARY,
 'F2_outcome_sensitivity':F2_OUTCOME_SENSITIVITY if BUILD_SENSITIVITY_REPAYMENT_FEATURES else [],
}
FORBIDDEN = {'ID','customer_id','tbl_loan_id','row_key','_source_row',
             'Total_Amount','target','due_date','Total_Amount_to_Repay',
             'Amount_Funded_By_Lender','Lender_portion_Funded','Lender_portion_to_be_repaid',
             'country_id','lender_id','New_versus_Repeat'}
for name, feats in FEATURE_SETS.items():
    assert not (set(feats)&FORBIDDEN), (name,set(feats)&FORBIDDEN)
    assert all(f in loans.columns for f in feats)
    assert len(set(feats))==len(feats)

# Preserve unencoded categories. Use one-hot in each CV training fold.
CATEGORICAL_PREDICTORS=['loan_type']
NUMERIC_PRIMARY=[f for f in F1_BORROWING_PRIMARY if f not in CATEGORICAL_PREDICTORS]
for fname,features in FEATURE_SETS.items():
    (OUT/(fname+'.json')).write_text(json.dumps(features,indent=2))
(OUT/'classification_features.json').write_text(json.dumps(F1_BORROWING_PRIMARY,indent=2))
(OUT/'regression_features.json').write_text(json.dumps(F1_BORROWING_PRIMARY,indent=2))
(OUT/'categorical_features.json').write_text(json.dumps(CATEGORICAL_PREDICTORS,indent=2))
(OUT/'numeric_features.json').write_text(json.dumps(NUMERIC_PRIMARY,indent=2))
print('Feature sets:',{k:len(v) for k,v in FEATURE_SETS.items()})


## 9 — Create task datasets and document exclusions

In [ ]:
# Date must exist for an application-time snapshot. Keep undated records in master for audit.
valid_date=loans._loan_date.notna()
classification=loans.loc[valid_date & loans.target.isin([0,1])].copy()
regression=loans.loc[valid_date & loans.Total_Amount.notna() & loans.Total_Amount.gt(0)].copy()

# Retain identifier/date/audit fields for grouping and splitting; NEVER pass them all to a model.
META=['row_key','ID','customer_id','tbl_loan_id','country_id','lender_id','disbursement_date','due_date','loan_type','duration']
cls_export=list(dict.fromkeys(META + F2_OUTCOME_SENSITIVITY + ['target']))
reg_export=list(dict.fromkeys(META + F2_OUTCOME_SENSITIVITY + ['Total_Amount']))
classification=classification[cls_export]
regression=regression[reg_export]
assert classification.row_key.is_unique and regression.row_key.is_unique
assert not classification.target.isna().any()
assert regression.Total_Amount.gt(0).all()

# Anti-leakage: do not export current classification target in regression task dataset,
# or current amount in classification task dataset.
assert 'target' not in regression.columns
assert 'Total_Amount' not in classification.columns

lifetime.to_csv(OUT/'customer_lifetime_profiles.csv',index=False)
loans.drop(columns=['_loan_date','_due_date']).to_csv(OUT/'loan_level_master.csv',index=False)
classification.to_csv(OUT/'classification_dataset.csv',index=False)
regression.to_csv(OUT/'regression_dataset.csv',index=False)

# Also export chosen primary model-only frames; ID and dates retained for CV audit.
cls_primary=classification[list(dict.fromkeys(['row_key','ID','customer_id','disbursement_date','due_date','target']+F1_BORROWING_PRIMARY))]
reg_primary=regression[list(dict.fromkeys(['row_key','ID','customer_id','disbursement_date','due_date','Total_Amount']+F1_BORROWING_PRIMARY))]
cls_primary.to_csv(OUT/'classification_primary_F1.csv',index=False)
reg_primary.to_csv(OUT/'regression_primary_F1.csv',index=False)
print('Dataset A:',len(lifetime),'Dataset B:',len(loans),'classification:',len(classification),'regression:',len(regression))


## 10 — Feature dictionary, source audit, missingness and research visuals

In [ ]:
DESCRIPTION = {
 'previous_loan_count':'Number of earlier loan events belonging to this borrower.',
 'previous_valid_amount_count':'Count of valid amounts among strictly earlier events.',
 'previous_total_amount_borrowed':'Sum of valid amounts on strictly earlier loans.',
 'previous_mean_loan_amount':'Mean disbursed principal on earlier loans.',
 'previous_min_loan_amount':'Minimum earlier disbursed principal.',
 'previous_max_loan_amount':'Maximum earlier disbursed principal.',
 'previous_mean_duration':'Mean duration of earlier loans with valid duration.',
 'days_since_previous_loan':'Days since most recent strictly earlier disbursement date.',
 'previous_loans_last_90d':'Prior loan events in [t-90days, t).',
 'previous_loans_last_180d':'Prior loan events in [t-180days, t).',
 'previous_mean_days_between_loans':'Mean gaps between unique earlier loan dates.',
 'previous_mean_amount_last_3':'Mean valid amounts in last 3 earlier loan records.',
 'previous_most_recent_loan_amount':'Loan amount of unique latest earlier loan date (otherwise undefined).',
 'previous_amount_trend_last_3':'Last minus first amount in three earlier loans with unambiguous dates.',
 'previous_known_outcome_count':'Due-date proxy eligible earlier outcome count; NOT confirmed resolution dates.',
 'previous_default_count':'Eligible earlier defaults under the due-date proxy.',
 'previous_repaid_count':'Eligible earlier repayments under the due-date proxy.',
 'previous_default_rate':'Eligible defaults / eligible outcomes; undefined if none.',
 'previous_repayment_rate':'Eligible repayments / eligible outcomes; undefined if none.',
 'days_since_last_default':'Days since last earlier default due-date, proxy only.',
 'most_recent_known_outcome':'Latest eligible proxy outcome if uniquely ordered.',
 'defaults_last_3_known_outcomes':'Defaults in last 3 eligible proxy outcomes.',
 'defaults_last_5_known_outcomes':'Defaults in last 5 eligible proxy outcomes.',
 'repayment_rate_last_3_known_outcomes':'Repaid share among last 3 eligible proxy outcomes.',
 'previous_loans_without_known_outcome':'Earlier borrow events not eligible as known outcomes.',
 'has_known_previous_outcome':'At least one outcome meets due-date proxy eligibility.',
 'has_previous_loan':'At least one strictly earlier loan event.',
 'has_previous_valid_amount':'At least one earlier loan has positive valid amount.',
 'has_previous_valid_duration':'At least one earlier loan has positive valid duration.',
 'has_multiple_previous_loan_dates':'At least two distinct prior disbursement dates.',
 'loan_type':'Proposed loan product; raw categorical value.',
 'duration':'Proposed term in days, if known at application.',
}
for c in CALENDAR_FEATURES:
    DESCRIPTION[c]='Calendar proxy derived from current disbursement date.'
feature_dict=pd.DataFrame([{
    'feature':f,
    'definition':DESCRIPTION.get(f,''),
    'group':('context' if f in CONTEXT_FEATURES else 'calendar' if f in CALENDAR_FEATURES
             else 'borrowing' if f in BORROW_FEATURES else 'history_availability'
             if f in BORROW_FLAGS else 'outcome_proxy'),
    'F0': f in F0_CONTEXT,'F1_primary':f in F1_BORROWING_PRIMARY,
    'F2_sensitivity':f in F2_OUTCOME_SENSITIVITY,
    'categorical':f in CATEGORICAL_PREDICTORS,
} for f in dict.fromkeys(F2_OUTCOME_SENSITIVITY)])
feature_dict.to_csv(OUT/'feature_dictionary.csv',index=False)
missingness=(loans[F2_OUTCOME_SENSITIVITY].isna().mean().sort_values(ascending=False)*100).rename_axis('feature').reset_index(name='missing_pct')
missingness.to_csv(AUDIT/'feature_missingness.csv',index=False)

# Auditable source feature-availability classification.
source_audit=pd.DataFrame([{
    'source_column':c,
    'role': ('source_identifier_no_model' if c in ID_COLS else
             'audit_only' if c in ['country_id','lender_id','New_versus_Repeat'] else
             'current_context_candidate' if c in CONTEXT_FEATURES else
             'target_classification' if c=='target' else
             'target_regression_and_prior_borrowing_only' if c=='Total_Amount' else
             'proxy_prediction_date' if c=='disbursement_date' else
             'prior_outcome_proxy_only' if c=='due_date' else 'excluded_post_decision_or_uncertain'),
} for c in REQUIRED])
source_audit.to_csv(AUDIT/'source_feature_audit.csv',index=False)

plt.rcParams.update({'figure.dpi':125,'savefig.dpi':200})
def fig_save(name):
    plt.tight_layout()
    plt.savefig(FIGURES/(name+'.png'),bbox_inches='tight')
    plt.savefig(FIGURES/(name+'.pdf'),bbox_inches='tight')
    plt.show()

fig,ax=plt.subplots(figsize=(6,3.7))
counts=clean.target.value_counts().reindex([0,1],fill_value=0)
ax.bar(['Repaid (0)','Default (1)'],counts.to_numpy())
ax.set_ylabel('Loan records');ax.set_title('Training source target distribution')
for i,v in enumerate(counts):ax.annotate(f'{v:,}',(i,v),ha='center',va='bottom')
fig_save('01_class_imbalance')

fig,ax=plt.subplots(figsize=(7,3.8))
amounts=clean.Total_Amount.dropna()
ax.hist(np.log1p(amounts),bins=45)
ax.set_xlabel('log1p(disbursed amount in KES)');ax.set_ylabel('Count')
ax.set_title('Distribution of disbursed principal')
fig_save('02_log_disbursement_distribution')

fig,ax=plt.subplots(figsize=(8,4.2))
show=missingness.head(18).iloc[::-1]
ax.barh(show.feature,show.missing_pct)
ax.set_xlabel('Missing (%)');ax.set_title('Missingness in derived history features')
fig_save('03_feature_missingness')

fig,ax=plt.subplots(figsize=(7,3.8))
prior=loans.previous_loan_count.dropna().clip(upper=10)
ax.hist(prior,bins=np.arange(-.5,11.5,1))
ax.set_xlabel('Prior loan events (10 = 10 or more)');ax.set_ylabel('Loans')
ax.set_title('Available history per loan snapshot')
fig_save('04_prior_loan_count')

monthly=(clean.dropna(subset=['disbursement_date']).assign(month=lambda d:d.disbursement_date.dt.to_period('M'))
         .groupby('month').agg(loans=('ID','size'),defaults=('target','sum')))
if len(monthly):
    fig,ax=plt.subplots(figsize=(9,3.9))
    monthly.loans.plot(ax=ax)
    ax.set_ylabel('Loan records');ax.set_xlabel('Disbursement month');ax.set_title('Observed loan volume over time')
    fig_save('05_loan_volume_over_time')
print('Saved figures:',len(list(FIGURES.glob('*.png'))))


## 11 — Metadata, fingerprints and final validation report

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):
            h.update(block)
    return h.hexdigest()
manifest={
  'input':'train.csv only',
  'input_path':str(TRAIN_PATH.resolve()),
  'input_sha256':sha256_file(TRAIN_PATH),
  'output_schema':'feature_eng_v2',
  'seed':SEED,
  'python':sys.version.split()[0],
  'pandas':pd.__version__,
  'numpy':np.__version__,
  'original_customer_ids_preserved':True,
  'six_digit_customer_ids_enforced':ENFORCE_SIX_DIGIT_CUSTOMER_IDS,
  'repeated_loan_id_policy':'quarantine_all_repeats',
  'same_day_loans_excluded_from_history':True,
  'prediction_time_proxy':'disbursement_date calendar date',
  'outcome_proxy':'prior due_date strictly before assessment; not verified actual resolution',
  'primary_uses_outcome_history':False,
  'primary_uses_country_or_lender':False,
  'categorical_encoding':'fit inside cross-validation training folds only',
  'feature_sets':FEATURE_SETS,
  'counts':quality | {'classification_rows':len(classification), 'regression_rows':len(regression),
                       'customer_profiles':len(lifetime)},
  'output_files':[p.name for p in OUT.glob('*.csv')],
  'note':'Prior V1 experiment scores and model binaries are not comparable without retraining.'
}
(OUT/'engineering_manifest_v2.json').write_text(json.dumps(manifest,indent=2,default=str))
assert set(classification.customer_id).issubset(set(source.customer_id))
assert set(regression.customer_id).issubset(set(source.customer_id))
assert not (set(F1_BORROWING_PRIMARY)&FORBIDDEN)
assert classification.row_key.is_unique and regression.row_key.is_unique
print('V2 engineering complete. Outputs:',OUT.resolve())
print('Review audits:',AUDIT.resolve())
print('Counts:',manifest['counts'])


## 12 — Example of correct CV-time categorical encoding (template; no training performed)

This is **illustrative only**. The full categorical `loan_type` is not one-hot encoded in exported CSVs: the encoder is learned on *training folds only*. The outer splits should be constructed and saved in Notebook 01 V2; GroupKFold and temporal CV must be compared. Training-fold resampling likewise must never touch validation data.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_cols=[c for c in F1_BORROWING_PRIMARY if c not in CATEGORICAL_PREDICTORS]
cat_cols=CATEGORICAL_PREDICTORS.copy()
preprocessor=ColumnTransformer(transformers=[
    ('numeric',Pipeline(steps=[('impute',SimpleImputer(strategy='median',add_indicator=True)),
                               ('scale',StandardScaler())]),num_cols),
    ('categorical',Pipeline(steps=[('impute',SimpleImputer(strategy='most_frequent')),
                                   ('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols),
])
print('Unfitted preprocessing template ready. Numerical:',len(num_cols),'categorical:',cat_cols)
print('Do NOT fit this transformer on all data before CV.')


## 13 — Presentation and modelling handoff checklist

- [ ] Verify `source_identifier_summary.csv` and `identity_mismatches.csv` (must be empty).
- [ ] Review `repeated_loan_ids_review.csv`, especially previously observed conflicting loan IDs; quarantined records may alter class prevalence.
- [ ] Confirm how many rows were excluded for missing dates and invalid regression target.
- [ ] Inspect first-loan and same-day chronology assertions.
- [ ] Explain why Dataset A is **descriptive** and Dataset B has historical snapshots.
- [ ] Primary training feature set is `F1_borrowing_primary.json`; F0 and F2 are comparison/sensitivity sets.
- [ ] Rebuild split assignments and retrain models from V2 data; **do not reuse V1 joblib models**.
- [ ] In the modelling pipeline, fit encoding/scaling/imputation *inside training folds*.
- [ ] The dataset has no transaction-level spending, income, affordability or actual label-availability timestamp.
- [ ] The ProSIT mentions backward `merge_asof` without exact matches; this notebook uses explicit per-customer strict earlier-date conditions, which achieve the chronology rule but are **not** literally `merge_asof`.
- [ ] FRED and Ghana-transfer inputs were deliberately excluded under the user's `train.csv`-only scope; report that departure from the original ProSIT.
